# Lab 07: Signal-Dependent Noise & the Speed-Accuracy Tradeoff**Computational Sensorimotor Control — Week 7****Task:** Apply signal-dependent noise (SDN) to the inverse dynamics controller from Week 6, demonstrate the speed-accuracy tradeoff, simulate Fitts' Law, and map activation variability.**Key concept:** Motor noise scales with command magnitude: σ(t) = k · |τ(t)|. This produces the speed-accuracy tradeoff and predicts Fitts' Law.**Primary task:** 6 cm semicircular arc (same as Week 6), with center-out reaches for Fitts' Law.

In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

from smc import (
    Arm, make_muscles, simulate_lambda,
    Q_REF, MUSCLE_DEFS,
    mass_matrix, coriolis, joint_accelerations, rk4_step,
    C_EXP, MU_LAMBDA,
)

arm = Arm()
ik  = arm.inverse_kinematics
fk  = arm.forward_kinematics
Jfn = arm.jacobian
start_hand = np.array(fk(Q_REF))

# ── Constants ──
R_ARC = 0.06
T_MOVE = 0.800
k_noise = 0.15  # SDN coefficient

# ── Muscle parameters ──
R_SH = np.array([m[3] for m in MUSCLE_DEFS])
R_EL = np.array([m[4] for m in MUSCLE_DEFS])
RL   = np.array([m[5] for m in MUSCLE_DEFS])
RHO  = np.array([m[1] for m in MUSCLE_DEFS])
K_PASS = np.array([m[2] for m in MUSCLE_DEFS])
ABS_R = np.abs(R_SH) + np.abs(R_EL)
R_mat = np.array([R_SH, R_EL])  # (2, 6) moment arm matrix

# ── Calcium filter / force-velocity constants (for EPH demo) ──
C_EXP_MM = 0.112; TAU_CA = 0.015
FV_F1 = 0.82; FV_F2 = 0.5; FV_F3 = 0.43; FV_F4 = 58.0

# ── Geometry helpers ──
def arc_geometry(R):
    center = start_hand + np.array([R, 0])
    tip    = start_hand + np.array([R, R])
    end    = start_hand + np.array([2*R, 0])
    return center, tip, end

def dense_arc(R, n=500):
    center = start_hand + np.array([R, 0])
    th = np.linspace(np.pi, 0, n)
    return np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def minjerk_basis(t, T):
    tau = np.clip(t / T, 0, 1)
    return 10*tau**3 - 15*tau**4 + 6*tau**5

def minjerk_arc(R, T=0.8, dt=0.001):
    center, _, _ = arc_geometry(R)
    t = np.arange(0, T + dt/2, dt)
    s = minjerk_basis(t, T)
    th = np.pi * (1 - s)
    pos = np.column_stack([center[0] + R*np.cos(th), center[1] + R*np.sin(th)])
    return t, pos

def minjerk_reach(target, T=0.8, dt=0.001):
    t = np.arange(0, T + dt/2, dt)
    s = minjerk_basis(t, T)
    pos = start_hand[None,:] + s[:,None] * (target - start_hand)[None,:]
    return t, pos

def id_torques(t_arr, pos, dt):
    n = len(t_arr)
    q   = np.array([ik(pos[i,0], pos[i,1]) for i in range(n)])
    qd  = np.gradient(q, dt, axis=0)
    qdd = np.gradient(qd, dt, axis=0)
    tau = np.zeros((n, 2))
    for i in range(n):
        M = mass_matrix(q[i]); C = coriolis(q[i], qd[i])
        tau[i] = M @ qdd[i] + C
    return q, qd, qdd, tau

MUSCLE_NAMES = ['Pec', 'BicL', 'BicS', 'Delt', 'TriL', 'TriLg']
NAVY = '#1B2A4A'; TEAL = '#2E86AB'; RED = '#E74C3C'; GREEN = '#27AE60'; GRAY = '#7F8C8D'

print("Setup complete. start_hand =", start_hand, "Q_REF =", np.degrees(Q_REF), "deg")

---## Part 1: Signal-Dependent Noise on EPH (Lecture §1 — Figure 1)In EPH, the brain sends R = (q₁*, q₂*) and co-contraction C. Signal-dependent noise acts on these three channels. The noisy commands propagate through the moment arm geometry, calcium filter, and force-velocity relationship to produce noisy hand trajectories.Below, helper functions implement the full EPH muscle model (calcium dynamics, force-velocity, passive springs). **Your task:** use `fwd_noisy_eph()` to run 200 noisy trials at T = 600 ms and T = 1200 ms on a 10 cm center-out reach, then reproduce Figure 1 from the lecture.

In [ ]:
# ── EPH helper functions (provided) ──
def muscle_lengths(q):
    return RL - R_SH*(q[0]-Q_REF[0]) - R_EL*(q[1]-Q_REF[1])

def muscle_velocities(qd):
    return -R_SH*qd[0] - R_EL*qd[1]

def lambda_from_RC(q_target, C):
    l_eq = RL - R_SH*(q_target[0]-Q_REF[0]) - R_EL*(q_target[1]-Q_REF[1])
    return l_eq - ABS_R * C

def eph_forces(q, qd, lam, ca, dt):
    l = muscle_lengths(q); v = muscle_velocities(qd)
    A = np.maximum(0, l - lam + MU_LAMBDA * v)
    mt = RHO * (np.exp(C_EXP_MM * A * 1000) - 1)
    mt = np.minimum(mt, 5000.0)
    for mi in range(6):
        mdd = (mt[mi] - ca[mi,0] - 2*TAU_CA*ca[mi,1]) / TAU_CA**2
        ca[mi,1] += mdd*dt; ca[mi,0] += ca[mi,1]*dt
        ca[mi,0] = max(ca[mi,0], 0.0)
    fv = FV_F1 + FV_F2 * np.arctan(FV_F3 + FV_F4 * v)
    F = ca[:,0] * fv + K_PASS * (l - RL)
    return R_mat @ F, A

def fwd_noisy_eph(qt_fn, C_fn, T, dt, k, rng):
    n = int(T/dt); q = np.zeros((n,2)); qd = np.zeros((n,2)); hand = np.zeros((n,2))
    ca = np.zeros((6,2)); q[0] = Q_REF.copy(); hand[0] = fk(Q_REF)
    for i in range(n-1):
        t = i*dt
        qt_nom = qt_fn(t); C_nom = C_fn(t)
        dq = qt_nom - Q_REF
        qt_n = qt_nom + k*np.abs(dq)*rng.standard_normal(2)
        C_n = max(0, C_nom + k*abs(C_nom)*rng.standard_normal())
        lam = lambda_from_RC(qt_n, C_n)
        tau, A = eph_forces(q[i], qd[i], lam, ca, dt)
        qdd = joint_accelerations(q[i], qd[i], tau)
        q[i+1] = q[i] + qd[i]*dt + 0.5*qdd*dt**2
        qd[i+1] = qd[i] + qdd*dt; hand[i+1] = fk(q[i+1])
    return hand

def make_eph_cmds(tgt, T, C=0.25):
    q_t = np.array(ik(tgt[0], tgt[1]))
    def qt_fn(t):
        if t < T: f = t/T; return (1-f)*Q_REF + f*q_t
        return q_t.copy()
    return qt_fn, lambda t: C

print("EPH helpers loaded.")

In [ ]:
# YOUR CODE HERE

# Reproduce the figure from the lecture


---## Part 2: Noisy Inverse Dynamics on the Arc (Lecture §2 — Figure 2)Now apply SDN to the inverse dynamics controller on the 6 cm arc from Week 6.**Your tasks:**1. Write `add_sdn(tau, k, rng)` — adds element-wise multiplicative noise to a 2D torque vector2. Write `forward_sim_noisy(q0, qd0, tau_nom, dt, k, rng)` — forward-simulates the arm with noisy torques3. Run 200 Monte Carlo trials and reproduce Figure 2 (3 panels)

In [ ]:
def add_sdn(tau, k, rng):
"""Add signal-dependent noise to a torque vector.
    
    Parameters
    ----------
    tau : ndarray, shape (2,)
        Nominal torque (shoulder, elbow).
    k : float
        Noise coefficient.
    rng : numpy Generator
        Random number generator.
    
    Returns
    -------
    tau_noisy : ndarray, shape (2,)
        Noisy torque.
    """
    # YOUR CODE HERE
    pass


In [ ]:
# YOUR CODE HERE

# Reproduce the figure from the lecture


---## Part 3: The Speed-Accuracy Tradeoff (Lecture §3 — Figures 3, 4)Vary the movement time on the same arc and measure how endpoint variability changes.**Your task:** Run 200 noisy trials at each of T = [400, 600, 800, 1000, 1200] ms and reproduce Figures 3 and 4.**Discussion Q:** The lecture states that endpoint SD scales as ~1/T. Do your data support this? Try plotting SD × T vs. T — if the scaling holds, this product should be approximately constant.

In [ ]:
# YOUR CODE HERE

# Reproduce the figure from the lecture


---## Part 4: Minimum-Variance — Smooth vs. Abrupt (Lecture §4 — Figure 5A–B)Compare the endpoint scatter for two trajectory shapes on a 10 cm center-out reach:- **Min-jerk:** smooth acceleration profile (low peak torque)- **Constant-velocity with 10% cosine ramps:** concentrates acceleration at the start and end (high peak torque)**Your task:** Implement `cosine_ramp_reach()`, compute ID torques for both, run 300 noisy trials, and compare.**Discussion Q:** The peak torque ratio is ~3×. Is the SD ratio also ~3×? Why or why not?

In [ ]:
def cosine_ramp_reach(target, T=0.8, dt=0.001, ramp_frac=0.10):
"""Constant-velocity reach with raised-cosine ramps."""
    # YOUR CODE HERE
    pass


---## Part 5: Simulating Fitts' Law (Lecture §4 — Figure 5C)For each combination of reach distance D and target width W, find the minimum movement time T that achieves at least 95% hits.**Your task:** Fill in the search loop below and plot MT vs. Index of Difficulty.**Discussion Q:** Human Fitts' Law slopes are 100–300 ms/bit. How does your simulated slope compare?

In [ ]:
# YOUR CODE HERE

# Reproduce the figure from the lecture


---## Part 6: Activation Variability on the Arc (Lecture §5 — Figure 7)Map the torque noise into muscle space via the moment arm matrix. For each muscle i:**noise_SD_i(t) = |r_sh_i| · k · |τ_sh(t)| + |r_el_i| · k · |τ_el(t)|**This is a direct linear mapping — not static optimization.**Your task:** Compute the activation noise heatmap and identify which muscles carry the most noise.**Discussion Q:** How does the noise pattern relate to the deterministic activation pattern from Lab06 Part 6?

In [ ]:
# ── Part 6: Activation variability heatmap ──
n_arc = len(tau_arc)
act_noise_sd = np.zeros((n_arc, 6))
for mi in range(6):
    act_noise_sd[:, mi] = np.abs(R_SH[mi]) * k_noise * np.abs(tau_arc[:,0]) + \
                           np.abs(R_EL[mi]) * k_noise * np.abs(tau_arc[:,1])

mx = act_noise_sd.max() + 1e-10

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow((act_noise_sd / mx).T, aspect='auto', cmap='YlOrRd',
               extent=[0, T_MOVE*1000, -0.5, 5.5], origin='lower', vmin=0, vmax=1)
ax.set_yticks(range(6)); ax.set_yticklabels(MUSCLE_NAMES)
ax.set_xlabel('Time (ms)')
ax.set_title('Activation noise on the 6 cm arc (normalized SD)')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

# Which muscles carry the most noise?
peak_noise = act_noise_sd.max(axis=0)
for mi in range(6):
    print(f"  {MUSCLE_NAMES[mi]:5s}: peak noise SD = {peak_noise[mi]:.4f} (normalized = {peak_noise[mi]/mx:.2f})")
print("\nShoulder muscles carry the most noise — they have the largest moment arms × largest torques.")

---## Summary1. **Signal-dependent noise** scales with command magnitude: σ = k · |τ| (Part 1–2)2. **The speed-accuracy tradeoff** emerges because faster movements require larger torques (Part 3)3. **Smooth trajectories minimize noise** by minimizing peak commands (Part 4)4. **Fitts' Law** emerges from SDN + the requirement to hit a finite-width target (Part 5)5. **Activation noise** is concentrated at muscles with the largest moment arms and torque contributions (Part 6)**Next week:** Module 3 begins — the visual system, sensory delays, and the first steps toward feedback control.